# Multi-task Gaussian Process

この Notebook では `MultiTaskGP` と `KroneckerMultiTaskGP` を比較します。

- `MultiTaskGP` は **long-format** データを使い、task index を入力特徴量として含めます。
- `KroneckerMultiTaskGP` は **block design** を使い、すべての task を同じ `X` で観測し、`Y` の各列が task に対応します。

task ごとに観測位置が異なる場合は `MultiTaskGP`、すべての task を同じ設計点で測定している場合は `KroneckerMultiTaskGP` が主な候補です。

## 1. Import と再現性設定

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.fit import fit_gpytorch_mll

from robotorchan.models import KroneckerMultiTaskGP, MultiTaskGP

torch.manual_seed(0)
dtype = torch.double

## 2. 相関を持つ2つの合成 task

In [ ]:
def task0(x: torch.Tensor) -> torch.Tensor:
    return torch.sin(2 * torch.pi * x)

def task1(x: torch.Tensor) -> torch.Tensor:
    return 0.7 * torch.sin(2 * torch.pi * x + 0.35) + 0.25

x0 = torch.linspace(0.05, 0.95, 10, dtype=dtype).unsqueeze(-1)
x1 = torch.linspace(0.10, 0.90, 7, dtype=dtype).unsqueeze(-1)

y0 = task0(x0) + 0.03 * torch.randn_like(x0)
y1 = task1(x1) + 0.03 * torch.randn_like(x1)

## 3. `MultiTaskGP`: long-format 表現

task feature を最後の列に追加します。ここでは `MultiTaskGP` の特徴を示すため、task 0 と task 1 で意図的に異なる `x` の観測位置を使います。

In [ ]:
task0_col = torch.zeros(len(x0), 1, dtype=dtype)
task1_col = torch.ones(len(x1), 1, dtype=dtype)

train_X_long = torch.cat(
    [
        torch.cat([x0, task0_col], dim=-1),
        torch.cat([x1, task1_col], dim=-1),
    ],
    dim=0,
)
train_Y_long = torch.cat([y0, y1], dim=0)

print("train_X_long:", train_X_long.shape)
print("train_Y_long:", train_Y_long.shape)
print(train_X_long[:3])

## 4. `MultiTaskGP` の学習

In [ ]:
mt_model = MultiTaskGP(
    train_X=train_X_long,
    train_Y=train_Y_long,
    task_feature=1,
)

print("raw_train_X:", mt_model.raw_train_X.shape)
print("raw_train_Y:", mt_model.raw_train_Y.shape)
print("supports_mll:", mt_model.supports_mll)

mt_mll = mt_model.make_mll()
fit_gpytorch_mll(mt_mll)

## 5. 両 task の posterior

posterior に渡す `X` から task feature を省略した場合は、`output_indices` で取得する task を指定できます。

In [ ]:
test_X = torch.linspace(0.0, 1.0, 200, dtype=dtype).unsqueeze(-1)

mt_model.eval()
with torch.no_grad():
    mt_post = mt_model.posterior(test_X, output_indices=[0, 1])
    mt_mean = mt_post.mean
    mt_var = mt_post.variance

print("posterior mean shape:", mt_mean.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(x0.squeeze(-1), y0.squeeze(-1), label="task 0 observations")
ax.scatter(x1.squeeze(-1), y1.squeeze(-1), label="task 1 observations")
ax.plot(test_X.squeeze(-1), mt_mean[..., 0].squeeze(-1), label="task 0 posterior")
ax.plot(test_X.squeeze(-1), mt_mean[..., 1].squeeze(-1), label="task 1 posterior")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("MultiTaskGP: long-format data")
ax.legend()
plt.show()

## 6. `KroneckerMultiTaskGP`: block-design 表現

Kronecker model では、すべての task を同じ設計点で観測する必要があります。そのため `train_X` は設計変数だけを持ち、`train_Y` は `(n, m)` の形で各列が task に対応します。

In [ ]:
train_X_block = torch.linspace(0.05, 0.95, 12, dtype=dtype).unsqueeze(-1)
train_Y_block = torch.cat(
    [task0(train_X_block), task1(train_X_block)],
    dim=-1,
)
train_Y_block = train_Y_block + 0.03 * torch.randn_like(train_Y_block)
print("train_X_block:", train_X_block.shape)
print("train_Y_block:", train_Y_block.shape)

## 7. `KroneckerMultiTaskGP` の学習

In [ ]:
kron_model = KroneckerMultiTaskGP(
    train_X=train_X_block,
    train_Y=train_Y_block,
)
print("raw_train_X:", kron_model.raw_train_X.shape)
print("raw_train_Y:", kron_model.raw_train_Y.shape)
print("raw_train_Yvar:", kron_model.raw_train_Yvar)
print("supports_mll:", kron_model.supports_mll)
kron_mll = kron_model.make_mll()
fit_gpytorch_mll(kron_mll)

## 8. Kronecker posterior

In [ ]:
kron_model.eval()
with torch.no_grad():
    kron_post = kron_model.posterior(test_X)
    kron_mean = kron_post.mean
print("posterior mean shape:", kron_mean.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(train_X_block.squeeze(-1), train_Y_block[:, 0], label="task 0 observations")
ax.scatter(train_X_block.squeeze(-1), train_Y_block[:, 1], label="task 1 observations")
ax.plot(test_X.squeeze(-1), kron_mean[..., 0], label="task 0 posterior")
ax.plot(test_X.squeeze(-1), kron_mean[..., 1], label="task 1 posterior")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("KroneckerMultiTaskGP: block design")
ax.legend()
plt.show()

## 9. 2モデルの使い分け

| モデル | 学習データ形式 | task ごとに異なる X | 主な用途 |
|---|---|---:|---|
| `MultiTaskGP` | long format、task index を `X` に含める | 可 | task ごとに異なる位置で観測 |
| `KroneckerMultiTaskGP` | `X[n,d]`, `Y[n,m]` | 不可 | 全 task を同じ設計点で観測 |

どちらの wrapper もコンストラクタに渡した raw tensor を保持し、`make_mll()` を提供します。